# LeetCode 394 — Decode String bằng Python

Link đề: https://leetcode.com/problems/decode-string/ · Độ khó: **Medium** · Chủ đề: Stack, Recursion, String

Skeleton LeetCode:

```python
class Solution:
    def decodeString(self, s: str) -> str:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: giải mã chuỗi dạng `k[encoded_string]` (lặp `encoded_string` đúng `k` lần), kể cả khi các cặp ngoặc **lồng nhau**.
2. Nắm các công cụ Python cần dùng: nhân chuỗi `"ab" * 3`, `str.isdigit()`, ghép số nhiều chữ số `num = num * 10 + int(c)`, `list` làm **ngăn xếp**, `re.sub` với hàm thay thế, và **đệ quy** với con trỏ chỉ số.
3. Đi từ cách **thay thế lặp đi lặp lại cặp ngoặc trong cùng** (regex) → **hai ngăn xếp** (số lặp và chuỗi đã có) → **đệ quy xuống từng cặp ngoặc** → **một ngăn xếp ký tự** (đẩy tất cả, khi gặp `]` thì gỡ ra).
4. Hiểu mấu chốt: ngoặc lồng nhau chính là cấu trúc **LIFO** (vào sau ra trước). Gặp `[` thì "cất" ngữ cảnh ngoài và bắt đầu chuỗi mới; gặp `]` thì "lấy" ngữ cảnh ra và ghép `trước + chuỗi_trong * k`.
5. Tránh các lỗi kinh điển: chỉ đọc số một chữ số (`10[a]`), quên đặt lại `num`/`res` sau `[`, ghép sai thứ tự `trước` / `trong`, dùng regex tham lam `.*`.

## 2. Đề bài & ví dụ

Cho chuỗi đã mã hóa `s`, hãy trả về chuỗi giải mã.

**Quy tắc mã hóa:** `k[encoded_string]` nghĩa là `encoded_string` được lặp lại đúng `k` lần, `k` là số nguyên dương. Có thể giả sử đầu vào luôn hợp lệ: không có khoảng trắng thừa, ngoặc vuông khớp nhau. Dữ liệu gốc **không chứa chữ số**; chữ số chỉ dùng làm số lần lặp `k` (nên không có đầu vào như `3a` hay `2[4]`). Độ dài kết quả không vượt quá `10^5`.

**Ví dụ 1:** `s = "3[a]2[bc]"` → `"aaabcbc"`.

**Ví dụ 2:** `s = "3[a2[c]]"` → `"accaccacc"`.

**Ví dụ 3:** `s = "2[abc]3[cd]ef"` → `"abcabccdcdcdef"`.

**Ràng buộc:** `1 <= s.length <= 30`; `s` gồm chữ cái thường, chữ số và `[`, `]`; `s` hợp lệ; mọi số nguyên trong `s` thuộc `[1, 300]`.

In [1]:
examples = [
    ("3[a]2[bc]", "aaabcbc"),
    ("3[a2[c]]", "accaccacc"),
    ("2[abc]3[cd]ef", "abcabccdcdcdef"),
]
for s, expected in examples:
    print(repr(s), "->", repr(expected))

'3[a]2[bc]' -> 'aaabcbc'
'3[a2[c]]' -> 'accaccacc'
'2[abc]3[cd]ef' -> 'abcabccdcdcdef'


## 3. Kiến thức nền

### 3.1. Nhân chuỗi và nối chuỗi

Trong Python `"ab" * 3` cho `"ababab"` (lặp chuỗi), còn `+` nối chuỗi: `"x" + "ab"` là `"xab"`. Hai phép này là cốt lõi của bài: kết quả của `k[inner]` là `inner * k`, rồi nối với phần đứng trước bằng `+`.

In [2]:
print("ab" * 3)              # ababab
print("x" + "ab" * 2)        # xabab   (nhân trước, cộng sau)
print("c" * 1, repr("c" * 0))    # c ''

ababab
xabab
c ''


### 3.2. `str.isdigit()` và ghép số có nhiều chữ số

`c.isdigit()` cho `True` nếu ký tự `c` là chữ số `0-9`. Số lặp có thể gồm **nhiều chữ số** (`100[leetcode]`). Khi duyệt từng ký tự, ghép dần: `num = num * 10 + int(c)`. Ví dụ đọc `"123"`: `0 → 1 → 12 → 123`.

In [3]:
print("7".isdigit(), "a".isdigit(), "[".isdigit())     # True False False
num = 0
for c in "123":
    num = num * 10 + int(c)
    print("đọc", c, "-> num =", num)

True False False
đọc 1 -> num = 1
đọc 2 -> num = 12
đọc 3 -> num = 123


### 3.3. `list` làm ngăn xếp (stack)

`append(x)` đẩy lên đỉnh, `pop()` lấy đỉnh ra (và trả về nó), `stack[-1]` xem đỉnh. Vì lồng nhau là "vào sau, ra trước": ngoặc mở sau cùng sẽ được đóng trước. Có thể dùng **hai list song song** (một cho số, một cho chuỗi) hoặc **một list chứa cặp** `(số, chuỗi)`.

In [4]:
counts, prefixes = [], []
counts.append(3); prefixes.append("")
counts.append(2); prefixes.append("a")
print(counts, prefixes)                 # [3, 2] ['', 'a']
print(counts.pop(), prefixes.pop())     # 2 a   (lấy cặp trong cùng ra trước)
print(counts, prefixes)                 # [3] ['']

pairs = [(3, ""), (2, "a")]            # cách khác: list các tuple
k, prefix = pairs.pop()
print(k, repr(prefix))                  # 2 'a'

[3, 2] ['', 'a']
2 a
[3] ['']
2 'a'


### 3.4. `re.sub` với hàm thay thế

`re.sub(pattern, repl, s)` thay mọi chỗ khớp `pattern` trong `s`. Nếu `repl` là **hàm**, nó nhận một đối tượng `match` và trả về chuỗi thay thế. `m.group(1)` là nhóm bắt thứ nhất (số lặp), `m.group(2)` là nhóm thứ hai (nội dung). Mẫu `(\d+)\[([a-z]*)\]` khớp một cặp ngoặc **không chứa ngoặc con** (trong cùng).

In [5]:
import re

pattern = re.compile(r'(\d+)\[([a-z]*)\]')
print(pattern.sub(lambda m: m.group(2) * int(m.group(1)), "3[a]2[bc]"))      # aaabcbc
print(pattern.sub(lambda m: m.group(2) * int(m.group(1)), "3[a2[c]]"))       # 3[acc]  (chỉ gỡ lớp trong cùng)

aaabcbc
3[acc]


### 3.5. Đệ quy và con trỏ chỉ số

Hàm đệ quy có thể nhận **chỉ số hiện tại `i`** và trả về **cặp (kết quả, chỉ số mới)** để người gọi biết đã đọc tới đâu. Mô hình quen thuộc: "đọc cho đến khi gặp `]` hoặc hết chuỗi; gặp `[` thì gọi lại chính nó cho phần bên trong".

In [6]:
def read_until_close(s, i):
    """Đọc ký tự cho tới khi gặp ']' hoặc hết chuỗi; trả (đã_đọc, chỉ_số_dừng)."""
    out = []
    while i < len(s) and s[i] != ']':
        out.append(s[i])
        i += 1
    return ''.join(out), i


print(read_until_close("abc]def", 0))   # ('abc', 3)
print(read_until_close("abc]def", 4))   # ('def', 7)

('abc', 3)
('def', 7)


### 3.6. Vì sao `''.join(list)` tốt hơn `+=` trong vòng lặp

Mỗi lần `res += c` Python có thể tạo chuỗi mới. Để gom nhiều mảnh, ta bỏ vào list rồi `''.join(list)` một lần (cho độ dài kết quả tới `10^5` thì cả hai đều ổn, nhưng `join` là thói quen tốt).

In [7]:
parts = []
for c in "abc":
    parts.append(c * 2)
print(parts, ''.join(parts))             # ['aa', 'bb', 'cc'] aabbcc

['aa', 'bb', 'cc'] aabbcc


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Giải mã `3[a2[c]]` bằng tay, **từ trong ra ngoài**:

1. Cặp ngoặc trong cùng là `2[c]` → `cc`. Chuỗi trở thành `3[acc]`.
2. Cặp `3[acc]` → `accaccacc`. Xong.

Nhận xét: cặp ngoặc nào **không còn ngoặc con** thì giải mã được ngay; sau đó nó biến thành chữ thường, nhường cho cặp bên ngoài. Cấu trúc này chính xác là **ngăn xếp**: cặp mở sau cùng đóng đầu tiên.

Ví dụ 3, `2[abc]3[cd]ef`: hai cặp ngoặc **cạnh nhau** (không lồng) giải mã độc lập: `abcabc` + `cdcdcd` + `ef`.

Nhớ thêm: số lặp có thể nhiều chữ số: `10[a]` → `aaaaaaaaaa` (đừng chỉ đọc chữ số `0`!).

In [8]:
# "đóng vai": giải mã tay 3[a2[c]] bằng hai bước thay thế
step0 = "3[a2[c]]"
step1 = step0.replace("2[c]", "c" * 2)       # gỡ cặp trong cùng
print(step1)                                  # 3[acc]
step2 = step1.replace("3[acc]", "acc" * 3)
print(step2)                                  # accaccacc

3[acc]
accaccacc


## 5. Cách 1 — Brute Force: lặp lại việc gỡ cặp ngoặc trong cùng bằng regex

### Ý tưởng

- Mẫu `(\d+)\[([a-z]*)\]` khớp cặp ngoặc **không chứa ngoặc nào bên trong**, tức cặp trong cùng.
- Lặp: `while '[' in s:` thay mọi cặp khớp bằng `nội_dung * số`. Mỗi vòng gỡ **một lớp** lồng nhau. Hết ngoặc thì dừng.

Đây là cách "làm đúng như tay" và rất dễ tin tưởng.

In [9]:
INNER = re.compile(r'(\d+)\[([a-z]*)\]')


def decode_regex(s):
    while '[' in s:
        s = INNER.sub(lambda m: m.group(2) * int(m.group(1)), s)
    return s


for s, expected in examples:
    print(decode_regex(s), expected, decode_regex(s) == expected)

aaabcbc aaabcbc True
accaccacc accaccacc True
abcabccdcdcdef abcabccdcdcdef True


### Độ phức tạp & nút thắt

- Mỗi vòng quét lại **toàn bộ chuỗi** và dựng chuỗi mới. Số vòng = độ lồng sâu `D`. Chuỗi trung gian có thể dài tới độ dài đáp án `L` (`≤ 10^5`). Tổng ≈ `O(D · L)` thời gian, `O(L)` bộ nhớ.
- Với giới hạn của bài vẫn chạy được, nhưng ta **quét lại** những phần đã giải mã xong và phụ thuộc vào mẫu regex (phải nhớ `[a-z]*` không có ngoặc con).

**Nút thắt:** nếu đi qua chuỗi **một lần** từ trái sang phải và *nhớ ngữ cảnh ngoài* bằng ngăn xếp thì mỗi ký tự chỉ xử lý một lần.

## 6. Ý tưởng thuật toán / bước đột phá

Đi từ trái sang phải, giữ **hai biến hiện tại**:

- `num`: số lặp đang đọc dở;
- `res`: chuỗi đã giải mã của **cấp ngoặc hiện tại** (phần nằm sau `[` gần nhất).

Gặp từng ký tự:

| Ký tự | Việc cần làm |
|---|---|
| chữ số | `num = num * 10 + int(c)` |
| chữ cái | `res += c` |
| `[` | **cất** `num` và `res` (là ngữ cảnh ngoài) lên ngăn xếp; đặt `num = 0`, `res = ''` để bắt đầu cấp mới |
| `]` | **lấy** ra `(k, prefix)`; kết quả cấp này là `prefix + res * k` |

Vì sao `prefix + res * k`? `prefix` là phần chữ đã có ở cấp ngoài *trước* ngoặc `[`; `res` là nội dung bên trong; sau khi lặp `k` lần phải *nối tiếp vào* `prefix`.

In [10]:
# Theo dõi từng ký tự của "3[a2[c]]" với hai ngăn xếp
num, res = 0, ''
s1, s2 = [], []
for c in "3[a2[c]]":
    if c.isdigit():
        num = num * 10 + int(c)
    elif c == '[':
        s1.append(num)
        s2.append(res)
        num, res = 0, ''
    elif c == ']':
        res = s2.pop() + res * s1.pop()
    else:
        res += c
    print(f"{c!r:4} num={num} res={res!r:10} s1={s1} s2={s2}")

'3'  num=3 res=''         s1=[] s2=[]
'['  num=0 res=''         s1=[3] s2=['']
'a'  num=0 res='a'        s1=[3] s2=['']
'2'  num=2 res='a'        s1=[3] s2=['']
'['  num=0 res=''         s1=[3, 2] s2=['', 'a']
'c'  num=0 res='c'        s1=[3, 2] s2=['', 'a']
']'  num=0 res='acc'      s1=[3] s2=['']
']'  num=0 res='accaccacc' s1=[] s2=[]


## 7. Các cách tiếp theo

Tổng cộng **4 cách**: (1) regex lặp; (2) hai ngăn xếp (cách chuẩn); (3) đệ quy với con trỏ chỉ số; (4) một ngăn xếp ký tự, gỡ khi gặp `]`.

### Cách 2 — Hai ngăn xếp (số và chuỗi)

Giải thích từng đoạn: đúng như bảng ở mục 6. `s1` chứa các số lặp đang chờ, `s2` chứa các `prefix` đang chờ. Ngăn xếp có độ sâu đúng bằng độ lồng hiện tại.

In [11]:
def decode_two_stacks(s):
    s1, s2 = [], []
    num, res = 0, ''
    for c in s:
        if c.isdigit():
            num = num * 10 + int(c)
        elif c == '[':
            s1.append(num)
            s2.append(res)
            num, res = 0, ''
        elif c == ']':
            res = s2.pop() + res * s1.pop()
        else:
            res += c
    return res


for s, expected in examples:
    print(decode_two_stacks(s), expected)

aaabcbc aaabcbc
accaccacc accaccacc
abcabccdcdcdef abcabccdcdcdef


**Code nộp LeetCode (Cách 2):**

In [12]:
class SolutionTwoStacks:
    def decodeString(self, s: str) -> str:
        s1, s2 = [], []
        num, res = 0, ''
        for c in s:
            if c.isdigit():
                num = num * 10 + int(c)
            elif c == '[':
                s1.append(num)
                s2.append(res)
                num, res = 0, ''
            elif c == ']':
                res = s2.pop() + res * s1.pop()
            else:
                res += c
        return res


print([SolutionTwoStacks().decodeString(s) for s, _ in examples])

['aaabcbc', 'accaccacc', 'abcabccdcdcdef']


Độ phức tạp: mỗi ký tự được xử lý một lần; công việc đáng kể là **dựng chuỗi kết quả** nên **Time `O(L)`** với `L` là tổng độ dài các chuỗi được tạo ra (chặn bởi kích thước đáp án và độ sâu: tối đa `O(D · L)` nếu lồng sâu, nhưng thực tế gần `O(L)`), **Space `O(L)`**.

### Cách 3 — Đệ quy theo từng cặp ngoặc

Định nghĩa `decode(i)` = *giải mã từ vị trí `i` cho tới khi gặp `]` (hoặc hết chuỗi)*, trả về `(chuỗi, chỉ số dừng)`.

- Gặp chữ số: gom số `k`.
- Gặp chữ cái: thêm vào kết quả.
- Gặp `[`: gọi `decode(i + 1)` để lấy nội dung bên trong và vị trí `]`; thêm `inner * k`; rồi bước qua `]`.
- Gặp `]`: dừng và trả về — đó là dấu hiệu kết thúc cấp hiện tại.

Ngăn xếp được thay bằng **ngăn xếp lời gọi hàm** của Python; mỗi cấp ngoặc là một lời gọi.

In [13]:
def decode_recursive(s):
    n = len(s)

    def parse(i):
        parts = []
        num = 0
        while i < n and s[i] != ']':
            c = s[i]
            if c.isdigit():
                num = num * 10 + int(c)
                i += 1
            elif c == '[':
                inner, i = parse(i + 1)         # i dừng ở dấu ']' tương ứng
                parts.append(inner * num)
                num = 0
                i += 1                           # bước qua ']'
            else:
                parts.append(c)
                i += 1
        return ''.join(parts), i

    return parse(0)[0]


for s, expected in examples:
    print(decode_recursive(s), expected)

aaabcbc aaabcbc
accaccacc accaccacc
abcabccdcdcdef abcabccdcdcdef


**Code nộp LeetCode (Cách 3):**

In [14]:
class SolutionRecursive:
    def decodeString(self, s: str) -> str:
        n = len(s)

        def parse(i):
            parts = []
            num = 0
            while i < n and s[i] != ']':
                c = s[i]
                if c.isdigit():
                    num = num * 10 + int(c)
                    i += 1
                elif c == '[':
                    inner, i = parse(i + 1)
                    parts.append(inner * num)
                    num = 0
                    i += 1
                else:
                    parts.append(c)
                    i += 1
            return ''.join(parts), i

        return parse(0)[0]


print([SolutionRecursive().decodeString(s) for s, _ in examples])

['aaabcbc', 'accaccacc', 'abcabccdcdcdef']


Độ phức tạp: **Time `O(L)`** (tương tự Cách 2), **Space** gồm độ sâu đệ quy `O(D)` (với `s.length ≤ 30` thì `D ≤ 7`; cũng an toàn với giới hạn đệ quy của Python khi `D` tới vài trăm) cộng bộ nhớ chuỗi `O(L)`.

### Cách 4 — Một ngăn xếp ký tự, gỡ khi gặp `]`

Ý tưởng khác: **đẩy mọi ký tự khác `]` vào ngăn xếp**. Khi gặp `]`:

1. Gỡ ra các phần tử cho tới khi gặp `[` — đó là nội dung bên trong (đảo lại cho đúng thứ tự).
2. Bỏ dấu `[`.
3. Gỡ tiếp các **chữ số** liền trước (đọc ngược) để ghép thành `k`.
4. Đẩy `nội_dung * k` (một mảnh chuỗi) trở lại ngăn xếp.

Cuối cùng nối toàn bộ ngăn xếp. Ưu điểm: chỉ **một cấu trúc dữ liệu**, nhìn thấy rõ "gỡ từ trong ra ngoài".

In [15]:
def decode_char_stack(s):
    stack = []
    for c in s:
        if c != ']':
            stack.append(c)
            continue
        parts = []
        while stack[-1] != '[':
            parts.append(stack.pop())
        stack.pop()                              # bỏ '['
        inner = ''.join(reversed(parts))
        digits = []
        while stack and stack[-1].isdigit():
            digits.append(stack.pop())
        k = int(''.join(reversed(digits)))
        stack.append(inner * k)
    return ''.join(stack)


for s, expected in examples:
    print(decode_char_stack(s), expected)

aaabcbc aaabcbc
accaccacc accaccacc
abcabccdcdcdef abcabccdcdcdef


**Code nộp LeetCode (Cách 4):**

In [16]:
class SolutionCharStack:
    def decodeString(self, s: str) -> str:
        stack = []
        for c in s:
            if c != ']':
                stack.append(c)
                continue
            parts = []
            while stack[-1] != '[':
                parts.append(stack.pop())
            stack.pop()
            inner = ''.join(reversed(parts))
            digits = []
            while stack and stack[-1].isdigit():
                digits.append(stack.pop())
            k = int(''.join(reversed(digits)))
            stack.append(inner * k)
        return ''.join(stack)


print([SolutionCharStack().decodeString(s) for s, _ in examples])

['aaabcbc', 'accaccacc', 'abcabccdcdcdef']


Độ phức tạp: **Time `O(L)`** (mỗi ký tự đẩy/gỡ một lần, cộng chi phí dựng chuỗi lặp), **Space `O(L)`**.

Lưu ý nhỏ: mảnh `inner * k` là **một phần tử chuỗi** nằm trong ngăn xếp; vì dữ liệu gốc không có chữ số nên mảnh này không bao giờ bị nhầm với số `k`.

## 8. TRACE từng bước

### Bảng trạng thái Cách 2 (hai ngăn xếp) với `"3[a2[c]]"`

| Ký tự | `num` | `res` | `s1` (số) | `s2` (chuỗi) | Ghi chú |
|---|---|---|---|---|---|
| `3` | 3 | `''` | `[]` | `[]` | gom số |
| `[` | 0 | `''` | `[3]` | `['']` | cất `(3, '')`, mở cấp mới |
| `a` | 0 | `'a'` | `[3]` | `['']` | thêm chữ |
| `2` | 2 | `'a'` | `[3]` | `['']` | gom số |
| `[` | 0 | `''` | `[3, 2]` | `['', 'a']` | cất `(2, 'a')`, mở cấp mới |
| `c` | 0 | `'c'` | `[3, 2]` | `['', 'a']` | thêm chữ |
| `]` | 0 | `'acc'` | `[3]` | `['']` | `res = 'a' + 'c' * 2` |
| `]` | 0 | `'accaccacc'` | `[]` | `[]` | `res = '' + 'acc' * 3` |

In [17]:
def trace_two_stacks(s):
    s1, s2 = [], []
    num, res = 0, ''
    for c in s:
        action = ""
        if c.isdigit():
            num = num * 10 + int(c)
            action = "gom số"
        elif c == '[':
            s1.append(num)
            s2.append(res)
            action = f"cất ({num}, {res!r}), mở cấp mới"
            num, res = 0, ''
        elif c == ']':
            k, prefix = s1[-1], s2[-1]
            action = f"res = {prefix!r} + {res!r} * {k}"
            res = s2.pop() + res * s1.pop()
        else:
            res += c
            action = "thêm chữ"
        print(f"{c!r:4} num={num} res={res!r:12} s1={s1!s:8} s2={s2}   # {action}")
    return res


print("kết quả:", trace_two_stacks("3[a2[c]]"))
print()
print("kết quả:", trace_two_stacks("2[abc]3[cd]ef"))

'3'  num=3 res=''           s1=[]       s2=[]   # gom số
'['  num=0 res=''           s1=[3]      s2=['']   # cất (3, ''), mở cấp mới
'a'  num=0 res='a'          s1=[3]      s2=['']   # thêm chữ
'2'  num=2 res='a'          s1=[3]      s2=['']   # gom số
'['  num=0 res=''           s1=[3, 2]   s2=['', 'a']   # cất (2, 'a'), mở cấp mới
'c'  num=0 res='c'          s1=[3, 2]   s2=['', 'a']   # thêm chữ
']'  num=0 res='acc'        s1=[3]      s2=['']   # res = 'a' + 'c' * 2
']'  num=0 res='accaccacc'  s1=[]       s2=[]   # res = '' + 'acc' * 3
kết quả: accaccacc

'2'  num=2 res=''           s1=[]       s2=[]   # gom số
'['  num=0 res=''           s1=[2]      s2=['']   # cất (2, ''), mở cấp mới
'a'  num=0 res='a'          s1=[2]      s2=['']   # thêm chữ
'b'  num=0 res='ab'         s1=[2]      s2=['']   # thêm chữ
'c'  num=0 res='abc'        s1=[2]      s2=['']   # thêm chữ
']'  num=0 res='abcabc'     s1=[]       s2=[]   # res = '' + 'abc' * 2
'3'  num=3 res='abcabc'     s1=[]       s2=[]   #

### Trace Cách 3 (đệ quy) với `"3[a2[c]]"`

Mỗi dòng thụt vào cho biết độ sâu của lời gọi `parse(i)`. Ta in "vào" và "ra" của mỗi lời gọi.

In [18]:
def trace_recursive(s):
    n = len(s)

    def parse(i, depth):
        pad = "  " * depth
        print(f"{pad}parse(i={i}) vào, s[i:]={s[i:]!r}")
        parts, num = [], 0
        while i < n and s[i] != ']':
            c = s[i]
            if c.isdigit():
                num = num * 10 + int(c)
                i += 1
            elif c == '[':
                inner, i = parse(i + 1, depth + 1)
                parts.append(inner * num)
                print(f"{pad}  gộp {inner!r} * {num} = {inner * num!r}")
                num = 0
                i += 1
            else:
                parts.append(c)
                i += 1
        out = ''.join(parts)
        print(f"{pad}parse trả về ({out!r}, i={i})")
        return out, i

    return parse(0, 0)[0]


print("kết quả:", trace_recursive("3[a2[c]]"))

parse(i=0) vào, s[i:]='3[a2[c]]'
  parse(i=2) vào, s[i:]='a2[c]]'
    parse(i=5) vào, s[i:]='c]]'
    parse trả về ('c', i=6)
    gộp 'c' * 2 = 'cc'
  parse trả về ('acc', i=7)
  gộp 'acc' * 3 = 'accaccacc'
parse trả về ('accaccacc', i=8)
kết quả: accaccacc


### Trace Cách 4 (một ngăn xếp ký tự) với `"3[a2[c]]"`

| Ký tự | Hành động | Ngăn xếp sau |
|---|---|---|
| `3` | đẩy | `['3']` |
| `[` | đẩy | `['3', '[']` |
| `a` | đẩy | `['3', '[', 'a']` |
| `2` | đẩy | `['3', '[', 'a', '2']` |
| `[` | đẩy | `['3', '[', 'a', '2', '[']` |
| `c` | đẩy | `['3', '[', 'a', '2', '[', 'c']` |
| `]` | gỡ `c` đến `[`; gỡ số `2`; đẩy `'cc'` | `['3', '[', 'a', 'cc']` |
| `]` | gỡ `cc`, `a` đến `[`: nội dung `'acc'`; gỡ số `3`; đẩy `'accaccacc'` | `['accaccacc']` |

In [19]:
def trace_char_stack(s):
    stack = []
    for c in s:
        if c != ']':
            stack.append(c)
            print(f"{c!r}: đẩy -> {stack}")
            continue
        parts = []
        while stack[-1] != '[':
            parts.append(stack.pop())
        stack.pop()
        inner = ''.join(reversed(parts))
        digits = []
        while stack and stack[-1].isdigit():
            digits.append(stack.pop())
        k = int(''.join(reversed(digits)))
        stack.append(inner * k)
        print(f"']': nội dung {inner!r}, k={k} -> đẩy {inner * k!r} -> {stack}")
    return ''.join(stack)


print("kết quả:", trace_char_stack("3[a2[c]]"))

'3': đẩy -> ['3']
'[': đẩy -> ['3', '[']
'a': đẩy -> ['3', '[', 'a']
'2': đẩy -> ['3', '[', 'a', '2']
'[': đẩy -> ['3', '[', 'a', '2', '[']
'c': đẩy -> ['3', '[', 'a', '2', '[', 'c']
']': nội dung 'c', k=2 -> đẩy 'cc' -> ['3', '[', 'a', 'cc']
']': nội dung 'acc', k=3 -> đẩy 'accaccacc' -> ['accaccacc']
kết quả: accaccacc


## 9. Code nộp LeetCode

Chọn **Cách 2** (hai ngăn xếp) làm bản nộp chính — ngắn, một lượt:

```python
class Solution:
    def decodeString(self, s: str) -> str:
        s1, s2 = [], []
        num, res = 0, ''
        for c in s:
            if c.isdigit():
                num = num * 10 + int(c)
            elif c == '[':
                s1.append(num)
                s2.append(res)
                num, res = 0, ''
            elif c == ']':
                res = s2.pop() + res * s1.pop()
            else:
                res += c
        return res
```

In [20]:
class Solution:
    def decodeString(self, s: str) -> str:
        s1, s2 = [], []
        num, res = 0, ''
        for c in s:
            if c.isdigit():
                num = num * 10 + int(c)
            elif c == '[':
                s1.append(num)
                s2.append(res)
                num, res = 0, ''
            elif c == ']':
                res = s2.pop() + res * s1.pop()
            else:
                res += c
        return res


for s, expected in examples:
    got = Solution().decodeString(s)
    print(repr(s), repr(got), got == expected)

'3[a]2[bc]' 'aaabcbc' True
'3[a2[c]]' 'accaccacc' True
'2[abc]3[cd]ef' 'abcabccdcdcdef' True


## 10. Vì sao đúng

**Bất biến (Cách 2):** sau khi xử lý xong `j` ký tự đầu, nếu các cặp ngoặc đang mở là `[` thứ nhất, thứ hai, ..., thứ `d` (từ ngoài vào trong) thì:
- `s1[t]` là số lặp ngay trước ngoặc mở thứ `t+1`, và `s2[t]` là phần chuỗi đã giải mã của cấp `t` ngay trước ngoặc đó;
- `res` là phần chuỗi đã giải mã của cấp ngoặc **trong cùng** hiện tại (cấp `d`), còn `num` là số đang đọc dở (nếu có).

**Khởi tạo:** chưa đọc gì, không ngoặc nào mở: `s1 = s2 = []`, `res = ''`, `num = 0`.

**Duy trì:**
- *chữ số:* ghép vào `num` — `num` luôn là số thập phân đang đọc (đề đảm bảo `k` luôn liền trước `[`).
- *chữ cái:* thuộc cấp hiện tại ⇒ `res += c`.
- *`[`:* mở một cấp mới. Cấp cũ (`res`) và số lặp (`num`) được cất; cấp mới bắt đầu rỗng.
- *`]`:* đóng cấp trong cùng. Theo định nghĩa `k[encoded]`, đóng góp của nó vào cấp ngoài là `res * k` và nó nối **tiếp theo** phần chữ `prefix` của cấp ngoài đã có trước ngoặc ⇒ `res_mới = prefix + res * k`. Cấp ngoài trở thành cấp trong cùng.

**Kết thúc:** hợp lệ nên mọi ngoặc đã đóng, ngăn xếp rỗng, `res` là chuỗi giải mã của cả đầu vào.

Cách 3 đúng theo cùng lý lẽ (ngăn xếp lời gọi thay thế ngăn xếp tay); Cách 4 đúng vì khi gặp `]` phần tử `[` gần nhất ngay phía dưới chính là ngoặc khớp (ngoặc lồng nhau đúng), và chữ số liền trước nó chính là `k` của cặp ấy.

## 11. Time/Space Complexity

Gọi `n = len(s)` (≤ 30), `L` = độ dài đáp án (≤ `10^5`), `D` = độ lồng sâu.

| Cách | Time | Space |
|---|---|---|
| 1. Regex lặp | `O(D · L)` | `O(L)` |
| 2. Hai ngăn xếp | `O(L)` thực tế (≤ `O(D · L)` lý thuyết) | `O(L)` |
| 3. Đệ quy | `O(L)` thực tế | `O(L + D)` |
| 4. Một ngăn xếp ký tự | `O(L)` thực tế | `O(L)` |

Bài này bị chặn bởi **kích thước đáp án** `L`: mọi cách đều phải tạo ra chuỗi dài `L` nên không thể tốt hơn `Ω(L)`. Phần "duyệt qua `s`" chỉ tốn `O(n)`.

## 12. Bộ test

### 12.1. Ví dụ đề + edge case

In [21]:
methods = [decode_regex, decode_two_stacks, decode_recursive, decode_char_stack,
           SolutionTwoStacks().decodeString, SolutionRecursive().decodeString,
           SolutionCharStack().decodeString, Solution().decodeString]

cases = [
    ("3[a]2[bc]", "aaabcbc"),
    ("3[a2[c]]", "accaccacc"),
    ("2[abc]3[cd]ef", "abcabccdcdcdef"),
    ("abc", "abc"),                              # không có ngoặc
    ("1[a]", "a"),                               # k = 1
    ("10[a]", "a" * 10),                         # số 2 chữ số
    ("100[ab]", "ab" * 100),                     # số 3 chữ số
    ("a2[b]c", "abbc"),                          # chữ ở trước và sau ngoặc
    ("2[a2[b2[c]]]", "abccbccabccbcc"),          # lồng 3 tầng
    ("2[2[2[a]]]", "a" * 8),
    ("2[a]2[b]2[c]", "aabbcc"),                  # ba cặp cạnh nhau
    ("ab2[c]de3[f]g", "abccdefffg"),
    ("3[z]2[2[y]pq4[2[jk]e1[f]]]ef", "zzzyypqjkjkefjkjkefjkjkefjkjkefyypqjkjkefjkjkefjkjkefjkjkefef"),
    ("300[a]", "a" * 300),
    ("10[10[10[10[10[a]]]]]", "a" * 100000),     # đáp án dài tối đa 10^5
    ("2[a2[b]c]d", "abbcabbcd"),
]
for s, expected in cases:
    for f in methods:
        got = f(s)
        assert got == expected, (f, s, got[:30], expected[:30])
print("OK:", len(cases), "test cố định x", len(methods), "cách")

OK: 16 test cố định x 8 cách


### 12.2. Sinh dữ liệu ngẫu nhiên có cấu trúc (oracle độc lập)

Ta sinh **cây cú pháp** trước: mỗi nút là danh sách các mục, mục là chữ cái hoặc `(k, nút_con)`. Từ cây ta:
- in ra chuỗi mã hóa (nối chữ và `k[...]`);
- tính đáp án đúng bằng cách **duyệt cây** và nhân chuỗi — hoàn toàn không dùng `[`, `]`, ngăn xếp hay regex.

In [22]:
import random

LETTERS = "abcxyz"


def gen_node(rng, depth, budget):
    """Trả về list các mục; mục là str (chữ cái) hoặc (k, node)."""
    items = []
    for _ in range(rng.randint(1, 3)):
        if depth < 3 and budget[0] > 0 and rng.random() < 0.45:
            k = rng.choice([1, 2, 3, 5, 9, 10, 12, 23, 100])
            budget[0] -= 1
            items.append((k, gen_node(rng, depth + 1, budget)))
        else:
            items.append("".join(rng.choice(LETTERS) for _ in range(rng.randint(1, 3))))
    return items


def encode(node):
    out = []
    for item in node:
        if isinstance(item, str):
            out.append(item)
        else:
            k, sub = item
            out.append(f"{k}[{encode(sub)}]")
    return "".join(out)


def expand(node):
    out = []
    for item in node:
        if isinstance(item, str):
            out.append(item)
        else:
            k, sub = item
            out.append(expand(sub) * k)
    return "".join(out)


rng = random.Random(394)
tested = 0
for _ in range(2500):
    tree = gen_node(rng, 0, [4])
    expected = expand(tree)
    if len(expected) > 100000:          # tuân thủ ràng buộc độ dài đáp án
        continue
    s = encode(tree)
    for f in methods:
        assert f(s) == expected, (f, s)
    tested += 1
print("OK:", tested, "đầu vào ngẫu nhiên, 8 cách đều khớp oracle (duyệt cây)")

OK: 2451 đầu vào ngẫu nhiên, 8 cách đều khớp oracle (duyệt cây)


### 12.3. Hiệu năng ở đáp án dài nhất

In [23]:
import time

big = "10[10[10[10[10[a]]]]]"                 # đáp án 100000 ký tự
big2 = "300[300[a]]"                          # 90000 ký tự
for name, f in [("regex", decode_regex), ("2 stack", Solution().decodeString),
                ("đệ quy", SolutionRecursive().decodeString), ("char stack", SolutionCharStack().decodeString)]:
    t0 = time.perf_counter()
    r1, r2 = f(big), f(big2)
    print(f"{name:10} {len(r1)} {len(r2)}  {time.perf_counter() - t0:.4f}s")
    assert len(r1) == 100000 and len(r2) == 90000

regex      100000 90000  0.0000s
2 stack    100000 90000  0.0000s
đệ quy     100000 90000  0.0000s
char stack 100000 90000  0.0000s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ đọc số một chữ số

In [24]:
def bug_single_digit(s):
    s1, s2 = [], []
    num, res = 0, ''
    for c in s:
        if c.isdigit():
            num = int(c)                          # SAI: không ghép nhiều chữ số
        elif c == '[':
            s1.append(num)
            s2.append(res)
            num, res = 0, ''
        elif c == ']':
            res = s2.pop() + res * s1.pop()
        else:
            res += c
    return res


print(repr(bug_single_digit("10[a]")), "(đúng là", repr(Solution().decodeString("10[a]")), ")")

'' (đúng là 'aaaaaaaaaa' )


**Vì sao sai:** `10` gồm hai chữ số: đọc `1` rồi bị ghi đè bởi `0`, nên `num = 0` và `res * 0` là chuỗi rỗng. **Sửa:** `num = num * 10 + int(c)`.

### Lỗi 2: quên đặt lại `num` (và `res`) sau `[` hoặc sau khi dùng

In [25]:
def bug_no_reset(s):
    s1, s2 = [], []
    num, res = 0, ''
    for c in s:
        if c.isdigit():
            num = num * 10 + int(c)
        elif c == '[':
            s1.append(num)
            s2.append(res)
            res = ''                              # SAI: quên num = 0
        elif c == ']':
            res = s2.pop() + res * s1.pop()
        else:
            res += c
    return res


print(repr(bug_no_reset("2[a]3[b]")), "(đúng là", repr(Solution().decodeString("2[a]3[b]")), ")")

'aabbbbbbbbbbbbbbbbbbbbbbb' (đúng là 'aabbb' )


**Vì sao sai:** sau `2[`, `num` vẫn là `2`. Đến cặp thứ hai, chữ số `3` được ghép thành `2*10+3 = 23`, nên `b` lặp 23 lần. **Sửa:** `num, res = 0, ''` ngay sau khi cất vào ngăn xếp.

### Lỗi 3: ghép sai thứ tự `prefix` và `res * k`

In [26]:
def bug_order(s):
    s1, s2 = [], []
    num, res = 0, ''
    for c in s:
        if c.isdigit():
            num = num * 10 + int(c)
        elif c == '[':
            s1.append(num)
            s2.append(res)
            num, res = 0, ''
        elif c == ']':
            res = res * s1.pop() + s2.pop()       # SAI: đặt phần lặp trước prefix
        else:
            res += c
    return res


print(repr(bug_order("a2[b]")), "(đúng là", repr(Solution().decodeString("a2[b]")), ")")
print(repr(bug_order("x2[yz]w")), "(đúng là", repr(Solution().decodeString("x2[yz]w")), ")")

'bba' (đúng là 'abb' )
'yzyzxw' (đúng là 'xyzyzw' )


**Vì sao sai:** `prefix` là phần chữ đứng **trước** ngoặc `[` nên phải đứng trước phần lặp: `prefix + res * k`. **Sửa:** `res = s2.pop() + res * s1.pop()`.

### Lỗi 4: dùng regex tham lam `.*`

In [27]:
def bug_greedy_regex(s):
    pattern = re.compile(r'(\d+)\[(.*)\]')       # SAI: .* tham lam, nuốt qua nhiều cặp ngoặc
    for _ in range(2):                           # chỉ lặp 2 vòng để quan sát (vòng 'while' thật sẽ không bao giờ dừng)
        s = pattern.sub(lambda m: m.group(2) * int(m.group(1)), s)
        print("   sau một vòng:", repr(s))
    return s


bug_greedy_regex("3[a]2[bc]")
print("đúng là", repr(Solution().decodeString("3[a]2[bc]")))

   sau một vòng: 'a]2[bca]2[bca]2[bc'
   sau một vòng: 'a]bca]2[bcabca]2[bca2[bc'
đúng là 'aaabcbc'


**Vì sao sai:** `.*` khớp **dài nhất có thể**, nên từ `3[` nó kéo đến tận `]` cuối cùng, gộp cả `a]2[bc` vào nội dung (và chuỗi có thể không bao giờ hết ngoặc nên vòng `while` lặp mãi — vì thế ví dụ trên chỉ chạy 2 vòng). Mẫu đúng phải **cấm ngoặc trong nội dung**: `[a-z]*` hoặc `[^\[\]]*`.

## 14. Mẹo & nhận dạng pattern

- Thấy **ngoặc lồng nhau, mở/đóng khớp cặp** ⇒ **ngăn xếp** (hoặc đệ quy).
- Mẫu chung của họ bài này: gặp `[` **cất ngữ cảnh**, gặp `]` **khôi phục ngữ cảnh và gộp** kết quả. Khác nhau chỉ ở "ngữ cảnh" là gì (ở đây: số lặp và chuỗi đã có).
- Hai trạng thái cần nhớ khi đọc số: `num` đang gom và `res` đang dựng; **đặt lại cả hai** khi mở cấp mới.
- Số nhiều chữ số: luôn dùng `num = num * 10 + digit`.
- Nếu phải phân tích biểu thức (có `+`, `*`, `()`) ⇒ cùng khung: ngăn xếp hoặc đệ quy xuống (recursive descent).
- Họ hàng: *Basic Calculator* (I, II, III), *Valid Parentheses*, *Remove Outermost Parentheses*, *Longest Absolute File Path*, *Mini Parser*, *Decoded String at Index* (LeetCode 880), *Number of Atoms*.

## 15. Biến thể

1. **Chỉ tính độ dài** chuỗi giải mã (không dựng chuỗi): ngăn xếp lưu `(k, độ_dài_prefix)`. Hữu ích khi đáp án khổng lồ.
2. **Đếm số lần xuất hiện của một chữ cái** trong chuỗi giải mã mà không dựng chuỗi.
3. **Tìm ký tự thứ `K`** của chuỗi giải mã (kiểu *Decoded String at Index*).
4. **Mã hóa ngược** (Encode String, LeetCode 471): nén chuỗi thành dạng `k[...]` ngắn nhất.
5. **`k` có thể bị thiếu** (`[ab]` nghĩa là một lần): nhánh `[` cần `num or 1`.

Minh họa biến thể 1:

In [28]:
def decoded_length(s):
    """Độ dài chuỗi giải mã mà không dựng chuỗi."""
    ks, prefixes = [], []
    num, length = 0, 0
    for c in s:
        if c.isdigit():
            num = num * 10 + int(c)
        elif c == '[':
            ks.append(num)
            prefixes.append(length)
            num, length = 0, 0
        elif c == ']':
            length = prefixes.pop() + length * ks.pop()
        else:
            length += 1
    return length


for s, expected in cases:
    assert decoded_length(s) == len(expected), s
print(decoded_length("10[10[10[10[10[a]]]]]"), decoded_length("300[300[300[a]]]"))   # 100000 27000000

100000 27000000


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Regex lặp | Gỡ cặp ngoặc trong cùng mỗi vòng | `O(D · L)` | `O(L)` | Hiểu bài, làm oracle nhanh |
| 2. Hai ngăn xếp | Cất `(k, prefix)` khi `[`, ghép khi `]` | `O(L)` | `O(L)` | **Cách nộp chuẩn**, ngắn gọn |
| 3. Đệ quy | Mỗi cặp ngoặc = một lời gọi, trả `(chuỗi, i)` | `O(L)` | `O(L + D)` | Khi ngữ pháp phức tạp hơn (recursive descent) |
| 4. Ngăn xếp ký tự | Đẩy hết, gỡ khi gặp `]` | `O(L)` | `O(L)` | Gần với cách làm tay, một cấu trúc dữ liệu |

## 17. Bài tập vận dụng (dễ → khó)

### Bài 1 (đoán output)

Đoán `guess_1()` trả về gì trước khi chạy.

In [29]:
def guess_1():
    num = 0
    for c in "305":
        num = num * 10 + int(c)
    return num, "ab" * 2 + "c" * 3


answer_1 = None   # TODO: điền một tuple (số, chuỗi)

### Bài 2 (tính tay)

Giải mã tay `"2[a3[b]]c"`: điền chuỗi kết quả vào `answer_2` (nhớ giải từ trong ra ngoài).

In [30]:
answer_2 = None   # TODO: điền chuỗi kết quả

### Bài 3 (điền TODO)

Hoàn thiện Cách 2 (hai ngăn xếp).

In [31]:
def decode_todo(s):
    s1, s2 = [], []
    num, res = 0, ''
    for c in s:
        if c.isdigit():
            # TODO 1: ghép chữ số vào num
            ...
        elif c == '[':
            # TODO 2: cất num và res; đặt lại num, res
            ...
        elif c == ']':
            # TODO 3: lấy ra và ghép: prefix + res * k
            ...
        else:
            res += c
    return res

### Bài 4 (tự viết lại)

Viết `count_letter(s, ch)` trả về số lần chữ cái `ch` xuất hiện trong chuỗi giải mã **mà không dựng chuỗi**. Ví dụ `count_letter("3[a2[c]]", "c") == 6`, `count_letter("3[a2[c]]", "a") == 3`. Gợi ý: giống `decoded_length` nhưng `length` chỉ tăng khi `c == ch`.

In [32]:
def count_letter(s, ch):
    # TODO: tự viết
    pass

### Bài 5 (biến thể / mini challenge)

Viết `kth_char(s, k)` trả về ký tự thứ `k` (đánh số từ `1`) của chuỗi giải mã. Bạn được phép dựng chuỗi (đáp án tối đa `10^5` ký tự). Mini challenge nâng cao (tùy chọn): làm mà **không dựng** chuỗi, dùng độ dài từng đoạn.

In [33]:
def kth_char(s, k):
    # TODO: tự viết
    pass

## 18. Gợi ý (3 mức)

**Bài 1**
- Mức 1: `num = num * 10 + int(c)` đọc ba chữ số `3, 0, 5` liên tiếp.
- Mức 2: `0 → 3 → 30 → 305`.
- Mức 3: `"ab" * 2 = "abab"`, `"c" * 3 = "ccc"`; ghép: `"ababccc"`. Đáp án `(305, 'ababccc')`.

**Bài 2**
- Mức 1: cặp trong cùng là `3[b]`.
- Mức 2: `3[b]` → `bbb`; còn `2[abbb]c`.
- Mức 3: `2[abbb]` → `abbbabbb`; cộng `c` ở cuối.

**Bài 3**
- Mức 1: `num = num * 10 + int(c)`.
- Mức 2: `s1.append(num); s2.append(res); num, res = 0, ''`.
- Mức 3: `res = s2.pop() + res * s1.pop()`.

**Bài 4**
- Mức 1: dùng hai list `ks`, `prefixes` như `decoded_length`, nhưng đếm số chữ `ch`.
- Mức 2: gặp chữ cái: `if c == ch: cnt += 1` (không phải `+= 1` vô điều kiện).
- Mức 3: khi gặp `]`: `cnt = prefixes.pop() + cnt * ks.pop()`.

**Bài 5**
- Mức 1: giải mã đủ rồi lấy `decoded[k - 1]`.
- Mức 2: nhớ chỉ số từ `1` nên trừ `1`.
- Mức 3: `return Solution().decodeString(s)[k - 1]`.

## 19. Đáp án tham khảo

In [34]:
# Bài 1
answer_1 = (305, 'ababccc')
assert guess_1() == answer_1

# Bài 2: 2[a3[b]]c -> 2[abbb]c -> abbbabbbc
answer_2 = "abbbabbbc"
assert answer_2 == Solution().decodeString("2[a3[b]]c")


# Bài 3
def decode_todo(s):
    s1, s2 = [], []
    num, res = 0, ''
    for c in s:
        if c.isdigit():
            num = num * 10 + int(c)
        elif c == '[':
            s1.append(num)
            s2.append(res)
            num, res = 0, ''
        elif c == ']':
            res = s2.pop() + res * s1.pop()
        else:
            res += c
    return res


for s, expected in cases:
    assert decode_todo(s) == expected


# Bài 4
def count_letter(s, ch):
    ks, prefixes = [], []
    num, cnt = 0, 0
    for c in s:
        if c.isdigit():
            num = num * 10 + int(c)
        elif c == '[':
            ks.append(num)
            prefixes.append(cnt)
            num, cnt = 0, 0
        elif c == ']':
            cnt = prefixes.pop() + cnt * ks.pop()
        elif c == ch:
            cnt += 1
    return cnt


assert count_letter("3[a2[c]]", "c") == 6
assert count_letter("3[a2[c]]", "a") == 3
for s, expected in cases:
    for ch in "abcz":
        assert count_letter(s, ch) == expected.count(ch), (s, ch)


# Bài 5
def kth_char(s, k):
    return Solution().decodeString(s)[k - 1]


assert kth_char("3[a2[c]]", 1) == "a" and kth_char("3[a2[c]]", 2) == "c" and kth_char("3[a2[c]]", 4) == "a"
print("Tất cả đáp án đã chạy đúng")

Tất cả đáp án đã chạy đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (hai ngăn xếp)

```python
s1, s2 = [], []            # số lặp, chuỗi prefix
num, res = 0, ''
for c in s:
    if c.isdigit():
        num = num * 10 + int(c)          # số nhiều chữ số
    elif c == '[':
        s1.append(num); s2.append(res)   # cất ngữ cảnh ngoài
        num, res = 0, ''                 # mở cấp mới
    elif c == ']':
        res = s2.pop() + res * s1.pop()  # prefix + (nội dung * k)
    else:
        res += c
return res
```

### Checklist

- [ ] Số lặp ghép bằng `num * 10 + int(c)` (xử lý `10[a]`, `100[ab]`).
- [ ] Sau `[` đặt lại **cả** `num` và `res`.
- [ ] Gặp `]`: `prefix + res * k` — đúng thứ tự (prefix đứng trước).
- [ ] Chữ cái nằm ngoài mọi ngoặc (`ab2[c]de`) vẫn được giữ.
- [ ] Không dùng regex tham lam `.*` cho ngoặc lồng nhau.
- [ ] Test: `"abc"`, `"1[a]"`, `"10[a]"`, `"2[a2[b2[c]]]"`, `"2[a]2[b]"`, đáp án dài `10^5`.